# Homogeneous elastic 2-D benchmark: FD3 / convFD3 / OPT3 / OPT5 / SPECFEM2D

The default case has no physical boundary condition: source and receivers are internal and comparison stops before any artificial-boundary return. Set `applyFreeSurface=true` later to restore the flat traction-free experiment. In 2-D plane strain the available displacement components are `uₓ` and `u_z`; there is no `u_y`.

In [ ]:
import Pkg
function find_flexopt_root(start=pwd())
    candidates = haskey(ENV, "FLEXOPT_ROOT") ? [ENV["FLEXOPT_ROOT"]] : String[]
    directory = abspath(start)
    while true
        push!(candidates, directory)
        parent = dirname(directory)
        parent == directory && break
        directory = parent
    end
    for candidate in unique(abspath.(expanduser.(candidates)))
        isfile(joinpath(candidate, "src", "flexOPT.jl")) && return candidate
    end
    error("Cannot locate flexOPT; set ENV[\"FLEXOPT_ROOT\"]")
end
flexopt_root = find_flexopt_root()
Pkg.activate(flexopt_root)
# Load the platform backend before flexOPT: batchGPU chooses CUDA, then
# Metal, and finally CPU. On this Apple machine it selects MetalBackend.
using Metal
include(joinpath(flexopt_root, "src", "batchFiles", "batchGPU.jl"))
backend isa KernelAbstractions.CPU && @warn(
    "No GPU backend is available; OPT recipes will be built on CPU")
@show backend
include(joinpath(flexopt_root, "src", "commonBatchs.jl"))
include(joinpath(flexopt_root, "src", "elasticWave2D.jl")) # ordinary FD explicit operators 
include(joinpath(flexopt_root, "src", "elasticGreens2D.jl")) # analytic solution
include(joinpath(flexopt_root, "src", "flexOPT.jl"))
include(joinpath(flexopt_root, "src", "specfemBenchmark.jl"))
using .commonBatchs, .elasticWave2D, .elasticGreens2D, .flexOPT, .specfemBenchmark
# CairoMakie renders static figures inline in IJulia/VS Code and can also
# render the MP4 frames below without opening a separate GLFW window.
using CairoMakie, Statistics, LinearAlgebra, SparseArrays, Base64, JLD2
CairoMakie.activate!(type="png")
@show VERSION Threads.nthreads() Base.active_project()


## Common physical model and experiment

In [ ]:
referenceDx = 500.0 # fixed physical source/SPECFEM reference scale
fdOptSpatialRefinement = 2 # set 1 for the former 500 m FD/OPT run
dx = referenceDx / fdOptSpatialRefinement
quickRun = true # fast 500 m diagnostic; false restores the 250 m comparison
optSpatialRefinement = 1 # FD3, convFD3 and OPT3 now share dx
buildHigherOrderRecipes = true # cache OPT4/OPT5 after coefficient audit
runOPT4 = false # enable only after ElasticLHSOperatorAudit passes
runOPT5 = false # enable only after ElasticLHSOperatorAudit passes
runOPTWithoutGamma = true # diagnostic: collocated force with same temporal weights
runConvFDBeyondBoundarySafeTime = false # boundary-truncation diagnostic only
waveformQuantity = :displacement # choose :velocity or :displacement
runSPECFEM2D = true # set false to reuse existing SPECFEM output files
waveformQuantity in (:velocity, :displacement) ||
    error("waveformQuantity must be :velocity or :displacement")
temporalRefinement = 2 # additional reduction of the OPT CFL time step
# Reduced homogeneous full space centred on the source. Its half-width
# is still checked below against the first possible P-wave edge arrival.
sourcePosition = (x=-10e3, z=-10e3)
domainHalfWidth = quickRun ? 35e3 : 72e3
x = collect((sourcePosition.x-domainHalfWidth):dx:
    (sourcePosition.x+domainHalfWidth))
applyFreeSurface = false
# In quick mode the arrival-time assertion below guarantees that no wave
# reaches the edge, so constructing padded Cerjan layers is unnecessary.
applyCerjan = !quickRun
z = collect((sourcePosition.z-domainHalfWidth):dx:
    (sourcePosition.z+domainHalfWidth))
nx, nz = length(x), length(z)
vp0, vs0, rho0 = 6000.0, 3464.0, 2700.0
material = applyFreeSurface ?
    repeat(reshape(z .<= 0.0, 1, :), nx, 1) : trues(nx, nz)
model = (
    ρ=fill(rho0 / 1e3, nx, nz),
    Vpv=fill(vp0 / 1e3, nx, nz),
    Vsv=fill(vs0 / 1e3, nx, nz),
)
cerjanCells = round(Int, 24 * referenceDx / dx)
cerjan = CerjanBoundarySpec(
    (cerjanCells, cerjanCells),
    (cerjanCells, 0);
    damping=0.0053 / fdOptSpatialRefinement^2,
)
activeCerjan = applyCerjan ? cerjan : nothing
bcFD = applyFreeSurface ?
    boundary_geometry(material, (dx, dx); cerjan=activeCerjan) :
    BoundaryConditionSet(free_surface=nothing, cerjan=activeCerjan,
        material_mask=BitArray(material), free_surface_mode=:pinned_void)
receiverZ = -5e3
receiverX = quickRun ? [-20e3, -15e3, -10e3, -5e3, 0.0] :
    [-20e3, -10e3, 0.0, 10e3, 20e3]
# A reusable 2-D receiver geometry. The original horizontal line remains
# the lightweight default for the multi-solver plots below.
receiverGridX = quickRun ? collect(-20e3:5e3:0.0) :
    collect(-30e3:10e3:30e3)
receiverGridZ = quickRun ? collect(-15e3:5e3:0.0) :
    collect(-30e3:10e3:10e3)
receiverGrid = [(x=xr, z=zr) for zr in receiverGridZ for xr in receiverGridX
    if hypot(xr - sourcePosition.x, zr - sourcePosition.z) >= 5e3]
receiverBoundaryMargins = [minimum((
    station.x - first(x), last(x) - station.x,
    station.z - first(z), last(z) - station.z,
)) for station in receiverGrid]
@assert minimum(receiverBoundaryMargins) >= (quickRun ? 20e3 : 30e3)
duration = quickRun ? 6.4 : 12.0
outputSampling = 0.10
# A three-point stencil needs a well-resolved shortest (S) wavelength.
# The source band is fixed by the reference 500 m grid. Refining OPT then
# measures discretization convergence without changing the physical source.
referencePointsPerSWavelength = 16
rickerSharpness = 1.5 # 1.0 is the former broader pulse
minimumAcceptedPointsPerSWavelength = 10
coarsestSpacing = referenceDx # preserve the physical Ricker band
sourceFrequency = rickerSharpness * vs0 /
    (referencePointsPerSWavelength * coarsestSpacing)
sourceDelay = 1.5 / sourceFrequency
sourceForce = 1.0e10 # N/m: total vertical line force in a unit-thickness slice
sourceMode = :gaussian # :point or :gaussian
sourceSpatialSigma = referenceDx # fixed while FD/OPT are refined
sourceCutoffSigmas = 2.0
sourceMode in (:point, :gaussian) || error("sourceMode must be :point or :gaussian")
function spatial_source_specs(xaxis, zaxis; center=sourcePosition,
    mode=sourceMode, sigma=sourceSpatialSigma, cutoff=sourceCutoffSigmas)
    if mode === :point
        return [(x=xaxis[argmin(abs.(xaxis .- center.x))],
            z=zaxis[argmin(abs.(zaxis .- center.z))], weight=1.0)]
    end
    specs = [(x=Float64(xv), z=Float64(zv),
        weight=exp(-((xv-center.x)^2 + (zv-center.z)^2) / (2sigma^2)))
        for xv in xaxis for zv in zaxis
        if hypot(xv-center.x, zv-center.z) <= cutoff*sigma]
    total = sum(s.weight for s in specs)
    [(x=s.x, z=s.z, weight=s.weight / total) for s in specs]
end
sourceSpecsReference = spatial_source_specs(x, z)
@assert isapprox(sum(s.weight for s in sourceSpecsReference), 1.0; atol=1e-14)
rickerSource(time) = begin
    a = π * sourceFrequency * (time - sourceDelay)
    (1 - 2a^2) * exp(-a^2)
end
pointsPerSWavelength = (
    FD3=vs0 / (sourceFrequency * dx),
    OPT3=vs0 / (sourceFrequency * (dx / optSpatialRefinement)),
    OPT5=vs0 / (sourceFrequency * (dx / optSpatialRefinement)),
)
distanceToNearestPhysicalEdge = minimum((
    sourcePosition.x - first(x), last(x) - sourcePosition.x,
    sourcePosition.z - first(z), last(z) - sourcePosition.z,
))
significantSourceStart = sourceDelay - 1 / sourceFrequency
earliestPEdgeArrival = significantSourceStart +
    distanceToNearestPhysicalEdge / vp0
@assert duration < earliestPEdgeArrival
@assert minimum(values(pointsPerSWavelength)) >=
    minimumAcceptedPointsPerSWavelength
@show applyFreeSurface applyCerjan (nx, nz) sourcePosition receiverZ
@show sourceFrequency sourceDelay rickerSharpness sourceForce sourceMode
@show sourceSpatialSigma length(sourceSpecsReference) cerjanCells
@show pointsPerSWavelength earliestPEdgeArrival
receiverGeometryFigure = Figure(size=(720, 620))
receiverGeometryAxis = Axis(receiverGeometryFigure[1, 1];
    xlabel="x (km)", ylabel="z (km)", aspect=DataAspect(),
    title="Available 2-D receiver grid (nearest-node sampling)")
scatter!(receiverGeometryAxis, getproperty.(receiverGrid, :x) ./ 1e3,
    getproperty.(receiverGrid, :z) ./ 1e3; label="receiver grid")
scatter!(receiverGeometryAxis, [sourcePosition.x / 1e3],
    [sourcePosition.z / 1e3]; marker=:star5, markersize=20,
    color=:gold, strokecolor=:black, label="source")
axislegend(receiverGeometryAxis)
display(receiverGeometryFigure)


## FD3: three points in space and time

In [ ]:
fdConfig = ElasticThreePointConfig2D(
    pointsInSpace=3, pointsInTime=3, supplementaryOrder=2, cfl=0.38,
)
fd = prepare_elastic_wave_2d(
    model, (dx, dx);
    material_mask=material,
    boundary_conditions=bcFD,
    config=fdConfig,
)
fdCoordinates = elastic_wave_coordinates(x, z, fd)
fdPadding = fd.padding
sourcePhysical = CartesianIndex(
    argmin(abs.(x .- sourcePosition.x)),
    argmin(abs.(z .- sourcePosition.z)),
)
sourceFD = sourcePhysical + CartesianIndex(Tuple(fdPadding[1, :]))
sourceSpecsFD = spatial_source_specs(fdCoordinates.x, fdCoordinates.z)
sourceNodesFD = [(index=CartesianIndex(
    argmin(abs.(fdCoordinates.x .- s.x)),
    argmin(abs.(fdCoordinates.z .- s.z))), weight=s.weight)
    for s in sourceSpecsFD]
fdSteps = ceil(Int, duration / fd.dt)
fdOutputStride = max(1, round(Int, outputSampling / fd.dt))
fdFramesX = Matrix{Float32}[copy(fd.ux)]
fdFramesZ = Matrix{Float32}[copy(fd.uz)]
fdTimes = Float64[0.0]
for step in 1:fdSteps
    step_elastic_wave_2d!(fd)
    for sourceNode in sourceNodesFD
        elasticWave2D.add_ricker_source!(
            fd, sourceNode.index;
            f0=sourceFrequency, t0=sourceDelay,
            amplitude=sourceForce * sourceNode.weight,
            component=:z, source_kind=:force,
        )
    end
    if step % fdOutputStride == 0 || step == fdSteps
        push!(fdFramesX, copy(fd.ux))
        push!(fdFramesZ, copy(fd.uz)); push!(fdTimes, fd.time)
    end
end
uxFD = cat(fdFramesX...; dims=3)
uzFD = cat(fdFramesZ...; dims=3)
@show fd.dt size(uzFD) maximum(abs, uxFD) maximum(abs, uzFD)


## convFD3, OPT3 and OPT5: construct the recipes and operators

In [ ]:
dxOPT = dx / optSpatialRefinement
xOPTPhysical = collect(first(x):dxOPT:last(x))
zOPTPhysical = collect(first(z):dxOPT:last(z))
solidOPT = applyFreeSurface ?
    repeat(reshape(zOPTPhysical .<= 0.0, 1, :),
        length(xOPTPhysical), 1) :
    trues(length(xOPTPhysical), length(zOPTPhysical))
dtOPT = 0.20 * dxOPT / (sqrt(2) * vp0 * temporalRefinement)
# Stable reference used by the previously working OPT benchmark. This is
# an exact change of coordinates x'=x/Δx, z'=z/Δz, t'=t/Δt, not a
# change of the physical model. The direct physical-recipe path remains
# available for conditioning experiments, but is not the benchmark default.
optFormulation = :scaled_coordinates # or :physical_recipe_experimental
if optFormulation === :scaled_coordinates
    recipeSpacing = (1.0, 1.0, 1.0)
    rhoOPT = Float64.(solidOPT)
    muOPT = fill(vs0^2 * (dtOPT / dxOPT)^2, size(solidOPT))
    lambdaOPT = fill((vp0^2 - 2vs0^2) * (dtOPT / dxOPT)^2, size(solidOPT))
elseif optFormulation === :physical_recipe_experimental
    recipeSpacing = (dxOPT, dxOPT, dtOPT)
    rhoOPT = fill(rho0, size(solidOPT))
    muOPT = fill(rho0 * vs0^2, size(solidOPT))
    lambdaOPT = fill(rho0 * (vp0^2 - 2vs0^2), size(solidOPT))
else
    error("unknown optFormulation=$optFormulation")
end
muOPT[.!solidOPT] .= 0.0
lambdaOPT[.!solidOPT] .= 0.0
optParameters = Dict{String,Any}(
    "famousEquationType" => "2DsismoTimeIsoHeteroSingleForce",
    "Δ" => recipeSpacing,
    "orderBtime" => 1, "orderBspace" => 1,
    "pointsInSpace" => 3, "pointsInTime" => 3,
    "supplementaryOrder" => 2,
    # Preserve all total-degree <= 2 Taylor conditions exactly; use
    # supplementaryOrder only to optimise the remaining null space.
    "taylorInverseMode" => :hierarchical_constrained,
    "fieldItpl" => (ptsSpace=1, ptsTime=1, offsetSpace=1.0,
        offsetTime=1, YorderBspace=-1, YorderBtime=-1),
    "materItpl" => (ptsSpace=1, ptsTime=1, offsetSpace=1.0,
        offsetTime=1, YorderBspace=-1, YorderBtime=-1),
    # The expensive semi-symbolic coefficient construction runs on the
    # GPU selected in the bootstrap cell (Metal here).
    "recipe_backend" => backend,
)
# Cache complete semi-symbolic recipes, not the very large prepared
# operators. The runtime Metal/CPU object is deliberately excluded from
# the cache key and is injected only when a recipe must be produced.
recipeCacheDirectory = "semiSymbolic"
recipeCacheVersion = 3 # hierarchical Taylor inverse; reject legacy recipes
function cachedOPTRecipe(parameters, prefix; existing=nothing)
    cacheParameters = Dict{String,Any}(
        key => value for (key, value) in parameters
        if key != "recipe_backend"
    )
    cacheParameters["recipe_cache_version"] = recipeCacheVersion
    function produceRecipe(config)
        runtimeParameters = Dict{String,Any}(config)
        pop!(runtimeParameters, "hash_id", nothing)
        pop!(runtimeParameters, "recipe_cache_version", nothing)
        runtimeParameters["recipe_backend"] = backend
        isnothing(existing) ?
            makeOPTsemiSymbolic(runtimeParameters) : existing
    end
    myProduceOrLoad(produceRecipe, cacheParameters,
        recipeCacheDirectory, prefix)
end
existingOPT3Recipe = @isdefined(optRecipe) ? optRecipe : nothing
optRecipe = cachedOPTRecipe(optParameters, "elastic2D_OPT3";
    existing=existingOPT3Recipe)

# Four/five-point spatial recipes; time remains a three-point stencil.
opt4Parameters = copy(optParameters)
opt4Parameters["pointsInSpace"] = 4
opt4Parameters["fieldItpl"] = merge(optParameters["fieldItpl"],
    (offsetSpace=1.5,))
opt4Parameters["materItpl"] = merge(optParameters["materItpl"],
    (offsetSpace=1.5,))
opt4Recipe = buildHigherOrderRecipes ? cachedOPTRecipe(
    opt4Parameters, "elastic2D_OPT4"; existing=nothing) : nothing

# Five-point OPT recipe: the time stencil remains three points so that
# only the spatial approximation changes relative to OPT3.
opt5Parameters = copy(optParameters)
# Validated OPT5 family used by seismo1Dbenchmark: five trial points
# retain the linear B-spline test order. orderBspace=3 is not OPT5 and
# generated a nearly singular first-step solve in this elastic problem.
opt5Parameters["orderBspace"] = 1
opt5Parameters["pointsInSpace"] = 5
opt5Parameters["fieldItpl"] = merge(optParameters["fieldItpl"],
    (offsetSpace=2.0,))
opt5Parameters["materItpl"] = merge(optParameters["materItpl"],
    (offsetSpace=2.0,))
# Keep OPT5 available, but do not even load/build it in the quick run.
# When enabled, never migrate the former invalid orderBspace=3 object.
opt5Recipe = buildHigherOrderRecipes ? cachedOPTRecipe(
    opt5Parameters, "elastic2D_OPT5"; existing=nothing) : nothing

# Collocation/delta-test recipe corresponding to the conventional FD3
# definition used in seismo1Dbenchmark.ipynb. Keep it beside OPT3: it is
# a coefficient reference, not an alias for the explicit flux solver.
conventionalFDParameters = copy(optParameters)
conventionalFDParameters["orderBspace"] = -1
conventionalFDParameters["orderBtime"] = -1
conventionalFDParameters["supplementaryOrder"] = 0
existingConvFD3Recipe = @isdefined(conventionalFDRecipe) ?
    conventionalFDRecipe : nothing
conventionalFDRecipe = cachedOPTRecipe(
    conventionalFDParameters, "elastic2D_convFD3";
    existing=existingConvFD3Recipe)
recipeComparison = (
    conventionalFD3=(orderBspace=-1, orderBtime=-1,
        supplementaryOrder=0, pointsInSpace=3, pointsInTime=3),
    OPT3=(orderBspace=optParameters["orderBspace"],
        orderBtime=optParameters["orderBtime"],
        supplementaryOrder=optParameters["supplementaryOrder"],
        pointsInSpace=optParameters["pointsInSpace"],
        pointsInTime=optParameters["pointsInTime"]),
    OPT4=(orderBspace=opt4Parameters["orderBspace"],
        orderBtime=opt4Parameters["orderBtime"],
        supplementaryOrder=opt4Parameters["supplementaryOrder"],
        pointsInSpace=opt4Parameters["pointsInSpace"],
        pointsInTime=opt4Parameters["pointsInTime"]),
    OPT5=(orderBspace=opt5Parameters["orderBspace"],
        orderBtime=opt5Parameters["orderBtime"],
        supplementaryOrder=opt5Parameters["supplementaryOrder"],
        pointsInSpace=opt5Parameters["pointsInSpace"],
        pointsInTime=opt5Parameters["pointsInTime"]),
)
# Public handles for inspecting the complete symbolic recipes in IJulia,
# e.g. `optRecipes.OPT5["recette"]`.
optRecipes = (convFD3=conventionalFDRecipe, OPT3=optRecipe,
    OPT4=opt4Recipe, OPT5=opt5Recipe)
@show recipeComparison
if runOPT4 || runOPT5
    coefficientGatePath = joinpath(flexopt_root, "data",
        "elastic_lhs_coefficient_gate.jld2")
    isfile(coefficientGatePath) || error(
        "Run ElasticLHSOperatorAudit.ipynb before OPT4/OPT5 propagation")
    coefficientGate = load(coefficientGatePath)
    @assert all(result.passed for result in
        values(coefficientGate["coefficient_health"]))
end


In [ ]:
optCerjan = CerjanBoundarySpec(
    (cerjan.lower[1] * optSpatialRefinement,
        cerjan.lower[2] * optSpatialRefinement),
    (cerjan.upper[1] * optSpatialRefinement, 0);
    damping=cerjan.damping / optSpatialRefinement^2,
)
activeOptCerjan = applyCerjan ? optCerjan : nothing
bcOPT = applyFreeSurface ?
    boundary_geometry(solidOPT, (dxOPT, dxOPT);
        free_surface_mode=:pinned_void, cerjan=activeOptCerjan) :
    BoundaryConditionSet(free_surface=nothing, cerjan=activeOptCerjan,
        material_mask=BitArray(solidOPT), free_surface_mode=:pinned_void)
modelsOPT = [rhoOPT, lambdaOPT, muOPT]
pointsOPT = getModelPoints(modelsOPT[1], 3,
    optRecipe["recette"].numbersOfTheSystem.numbersOfTheSystemL.timeMarching)
familyOPT = (models=modelsOPT, modelPoints=pointsOPT,
    Δ=recipeSpacing, modelName="homogeneous_OPT3_$(optFormulation)")
operatorTimings = Dict{Symbol,Float64}()
timedOPT3Assembly = @timed numericalOperatorConstruction(Dict{String,Any}(
    "optRec" => optRecipe, "modelFam" => familyOPT,
    "absorbingBoundaries" => nothing,
    "maskedRegionInSpace" => nothing,
    "boundaryConditions" => bcOPT,
    "representation" => "matrixfree",
))["numOperators"]
numericalVolume = timedOPT3Assembly.value
operatorTimings[:OPT3_assembly] = timedOPT3Assembly.time
timedOPT3Preparation = @timed prepareLinearSystem(numericalVolume;
    free_surface_spacing=recipeSpacing[1:2])
preparedVolume = timedOPT3Preparation.value
operatorTimings[:OPT3_preparation] = timedOPT3Preparation.time
preparedOPT4Volume = if runOPT4
    pointsOPT4 = getModelPoints(modelsOPT[1], 4,
        opt4Recipe["recette"].numbersOfTheSystem.numbersOfTheSystemL.timeMarching)
    familyOPT4 = (models=modelsOPT, modelPoints=pointsOPT4,
        Δ=recipeSpacing, modelName="homogeneous_OPT4_$(optFormulation)")
    numericalOPT4 = numericalOperatorConstruction(Dict{String,Any}(
        "optRec" => opt4Recipe, "modelFam" => familyOPT4,
        "absorbingBoundaries" => nothing, "maskedRegionInSpace" => nothing,
        "boundaryConditions" => bcOPT, "representation" => "matrixfree",
    ))["numOperators"]
    prepareLinearSystem(numericalOPT4; free_surface_spacing=recipeSpacing[1:2])
else
    nothing
end
preparedOPT5Volume = if runOPT5
    pointsOPT5 = getModelPoints(modelsOPT[1], 5,
        opt5Recipe["recette"].numbersOfTheSystem.numbersOfTheSystemL.timeMarching)
    familyOPT5 = (models=modelsOPT, modelPoints=pointsOPT5,
        Δ=recipeSpacing, modelName="homogeneous_OPT5_$(optFormulation)")
    numericalOPT5 = numericalOperatorConstruction(Dict{String,Any}(
        "optRec" => opt5Recipe, "modelFam" => familyOPT5,
        "absorbingBoundaries" => nothing,
        "maskedRegionInSpace" => nothing,
        "boundaryConditions" => bcOPT,
        "representation" => "matrixfree",
    ))["numOperators"]
    prepareLinearSystem(numericalOPT5;
        free_surface_spacing=recipeSpacing[1:2])
else
    nothing
end
timedConvFDAssembly = @timed numericalOperatorConstruction(Dict{String,Any}(
    "optRec" => conventionalFDRecipe, "modelFam" => familyOPT,
    "absorbingBoundaries" => nothing,
    "maskedRegionInSpace" => nothing,
    "boundaryConditions" => bcOPT,
    "representation" => "matrixfree",
))["numOperators"]
numericalConvFD = timedConvFDAssembly.value
operatorTimings[:convFD3_assembly] = timedConvFDAssembly.time
timedConvFDPreparation = @timed prepareLinearSystem(numericalConvFD;
    free_surface_spacing=recipeSpacing[1:2])
preparedConvFDVolume = timedConvFDPreparation.value
operatorTimings[:convFD3_preparation] = timedConvFDPreparation.time

# Optional traction recipe σn=0, assembled additively on surface rows.
if applyFreeSurface
boundaryParameters = copy(optParameters)
boundaryParameters["famousEquationType"] = "elasticTractionFree2D"
boundaryRecipe = cachedOPTRecipe(
    boundaryParameters, "elasticTractionFree2D_OPT3")
normalX = zeros(size(solidOPT)); normalZ = zeros(size(solidOPT))
for (point, normal) in zip(bcOPT.free_surface.points, bcOPT.free_surface.normals)
    normalX[point], normalZ[point] = normal
end
boundaryModels = [lambdaOPT, muOPT, normalX, normalZ]
boundaryPoints = getModelPoints(boundaryModels[1], 3,
    boundaryRecipe["recette"].numbersOfTheSystem.numbersOfTheSystemL.timeMarching)
boundaryFamily = (models=boundaryModels, modelPoints=boundaryPoints,
    Δ=recipeSpacing, modelName="homogeneous_free_surface")
numericalBoundary = numericalOperatorConstruction(Dict{String,Any}(
    "optRec" => boundaryRecipe, "modelFam" => boundaryFamily,
    "absorbingBoundaries" => (isnothing(bcOPT.cerjan) ? nothing : cerjan_padding(bcOPT.cerjan)),
    "maskedRegionInSpace" => bcOPT.free_surface.points,
    "representation" => "matrixfree",
))["numOperators"]
preparedBoundary = prepareLinearSystem(numericalBoundary)
surfaceWhole = numericalVolume.numericalOperators.left.geometry.freeSurfaceBoundary.points
    preparedOPT = overlapBoundaryLinearSystem(
    preparedVolume, preparedBoundary, surfaceWhole;
    mode=:additive, boundary_weight=1.0,
    )
    preparedConvFD = overlapBoundaryLinearSystem(
        preparedConvFDVolume, preparedBoundary, surfaceWhole;
        mode=:additive, boundary_weight=1.0,
    )
    preparedOPT4 = runOPT4 ? overlapBoundaryLinearSystem(
        preparedOPT4Volume, preparedBoundary, surfaceWhole;
        mode=:additive, boundary_weight=1.0) : nothing
    preparedOPT5 = runOPT5 ? overlapBoundaryLinearSystem(
        preparedOPT5Volume, preparedBoundary, surfaceWhole;
        mode=:additive, boundary_weight=1.0) : nothing
else
    preparedOPT = preparedVolume
    preparedConvFD = preparedConvFDVolume
    preparedOPT4 = runOPT4 ? preparedOPT4Volume : nothing
    preparedOPT5 = runOPT5 ? preparedOPT5Volume : nothing
end
@assert preparedOPT.NForceField == 2
@assert preparedConvFD.NForceField == 2
runOPT4 && @assert preparedOPT4.NForceField == 2
runOPT5 && @assert preparedOPT5.NForceField == 2
optOperatorUnitConvention = optFormulation
boundaryOverlapRows = hasproperty(preparedOPT, :boundary_overlap_rows) ?
    preparedOPT.boundary_overlap_rows : Int[]
@show quickRun optFormulation optSpatialRefinement dxOPT dtOPT recipeSpacing
@show operatorTimings
@show preparedOPT.spaceShape preparedConvFD.spaceShape runOPT5 length(boundaryOverlapRows)
runOPT5 && @show preparedOPT5.spaceShape


In [ ]:
# A homogeneous isotropic recipe must be invariant under x↔z together
# with uₓ↔u_z. Inspect central prepared coefficients before propagation.
function xz_recipe_symmetry(prepared)
    shape = prepared.spaceShape
    length(shape) == 2 && shape[1] >= 5 && shape[2] >= 5 ||
        error("the symmetry diagnostic needs a 2-D interior")
    nspace, nfield = prod(shape), prepared.NField
    nfield == 2 || error("elastic x↔z comparison needs two fields")
    point = CartesianIndex(cld(shape[1], 2), cld(shape[2], 2))
    lp0 = LinearIndices(shape)[point]
    row(expr) = expr + nfield * (lp0 - 1)
    function coefficients(matrix, equation)
        columns, values = SparseArrays.findnz(matrix[row(equation), :])
        result = Dict{NTuple{4,Int},Float64}()
        for (column, value) in zip(columns, values)
            time = cld(column, nspace * nfield)
            within_time = mod1(column, nspace * nfield)
            field = cld(within_time, nspace)
            lp = mod1(within_time, nspace)
            q = CartesianIndices(shape)[lp]
            offset = Tuple(q - point)
            result[(time, field, offset[1], offset[2])] = Float64(value)
        end
        result
    end
    swap_field(field) = 3 - field
    function mismatch(matrix)
        xrow, zrow = coefficients(matrix, 1), coefficients(matrix, 2)
        transformed_z = Dict((time, swap_field(field), dz, dx) => value
            for ((time, field, dx, dz), value) in zrow)
        keys_union = union(keys(xrow), keys(transformed_z))
        difference2 = sum((get(xrow, key, 0.0) -
            get(transformed_z, key, 0.0))^2 for key in keys_union)
        scale2 = sum(max(get(xrow, key, 0.0)^2,
            get(transformed_z, key, 0.0)^2) for key in keys_union)
        sqrt(difference2 / max(scale2, eps(Float64)))
    end
    (A_unknown=mismatch(prepared.A_unknown),
     L_known=mismatch(prepared.L_known), point)
end
recipeSymmetry = (
    OPT3=Base.invokelatest(xz_recipe_symmetry, preparedOPT),
    convFD3=Base.invokelatest(xz_recipe_symmetry, preparedConvFD),
)
if runOPT4
    recipeSymmetry = merge(recipeSymmetry,
        (OPT4=Base.invokelatest(xz_recipe_symmetry, preparedOPT4),))
end
if runOPT5
    recipeSymmetry = merge(recipeSymmetry,
        (OPT5=Base.invokelatest(xz_recipe_symmetry, preparedOPT5),))
end
symmetryMaximum = maximum(value for result in values(recipeSymmetry)
    for value in (result.A_unknown, result.L_known))
symmetryInterpretation = symmetryMaximum <= 1e-7 ?
    :excellent_discrete_xz_symmetry :
    symmetryMaximum <= 1e-4 ? :small_coefficient_asymmetry :
    :significant_coefficient_asymmetry
@show recipeSymmetry symmetryInterpretation
symmetryMaximum > 1e-4 && @warn(
    "Significant x↔z coefficient asymmetry", recipeSymmetry)
# Important: x↔z symmetry is only the square grid's 90° symmetry. It does
# not test rotational dispersion at intermediate propagation angles.


In [ ]:
optPadding = isnothing(bcOPT.cerjan) ? zeros(Int, 2, 2) : cerjan_padding(bcOPT.cerjan)
@assert optOperatorUnitConvention === optFormulation "restart the kernel and rerun the OPT operator cell"
xOPT = range(first(xOPTPhysical) - optPadding[1,1] * dxOPT;
    step=dxOPT, length=preparedOPT.spaceShape[1])
zOPT = range(first(zOPTPhysical) - optPadding[1,2] * dxOPT;
    step=dxOPT, length=preparedOPT.spaceShape[2])
sourceIXOPT = argmin(abs.(xOPTPhysical .- sourcePosition.x))
sourceIZOPT = argmin(abs.(zOPTPhysical .- sourcePosition.z))
sourceIndexOPT = CartesianIndex(
    sourceIXOPT + optPadding[1,1], sourceIZOPT + optPadding[1,2])
sourceLinearOPT = LinearIndices(preparedOPT.spaceShape)[sourceIndexOPT]
sourceSpecsOPT = spatial_source_specs(xOPTPhysical, zOPTPhysical)
sourceNodesOPT = [(linear=LinearIndices(preparedOPT.spaceShape)[
        CartesianIndex(argmin(abs.(xOPT .- s.x)),
            argmin(abs.(zOPT .- s.z)))], weight=s.weight)
    for s in sourceSpecsOPT]
@assert isapprox(sum(s.weight for s in sourceNodesOPT), 1.0; atol=1e-14)
optSteps = ceil(Int, duration / dtOPT)
optOutputStride = max(1, round(Int, outputSampling / dtOPT))
# Align the force stencil with (past..., future): for three time points,
# the first solve for u(+Δt) receives f(-Δt), f(0), f(+Δt).
nPastForceLevels = preparedOPT.timePointsUsedForOneStep - 1
sourceTimesOPT = ((1 - nPastForceLevels):optSteps) .* dtOPT
@assert sourceTimesOPT[preparedOPT.timePointsUsedForOneStep] == dtOPT
waveletOPT = rickerSource.(sourceTimesOPT)
# In SI, famousEquations uses f=ρa and the cell body-force density is
# sourceForce/(ΔxΔz). Under the scaled-coordinate PDE this becomes
# f'=f Δt²/ρ = sourceForce Δt²/(ρ ΔxΔz).
sourceScaleOPT = optFormulation === :scaled_coordinates ?
    sourceForce * dtOPT^2 / (rho0 * dxOPT^2) :
    sourceForce / dxOPT^2
@assert optFormulation !== :scaled_coordinates ||
    isapprox(sourceScaleOPT * rho0 * dxOPT^2 / dtOPT^2, sourceForce)
sourceOPT = zeros(Float64, preparedOPT.NforcePoints,
    preparedOPT.NForceField, length(sourceTimesOPT))
for sourceNode in sourceNodesOPT
    sourceOPT[sourceNode.linear, 2, :] .+=
        sourceNode.weight .* sourceScaleOPT .* waveletOPT
end
propagationOPT = propagateLinearSystem(
    preparedOPT, optSteps, dtOPT;
    sourceFull=sourceOPT, output_stride=optOutputStride,
    blowup_limit=1e8,
    solver_name="OPT3",
)
@assert !propagationOPT.stopped_early

# Controlled Γ diagnostic. Preserve, for every temporal source slot, the
# signed column sum of R_force but place it only on the collocated equation
# row. Thus only the spatial redistribution is removed.
function collocated_source_operator(prepared, source_nodes; force_field=2)
    rows=Int[]; columns=Int[]; values=Float64[]
    nspace, nfield = prepared.NpointsSpace, prepared.NField
    for node in source_nodes, time_slot in 1:prepared.timePointsUsedForOneStep
        column = node.linear + (force_field-1)*nspace +
            (time_slot-1)*nspace*prepared.NForceField
        coefficient = sum(@view prepared.R_force[:, column])
        push!(rows, force_field + nfield*(node.linear-1))
        push!(columns, column); push!(values, coefficient)
    end
    sparse(rows, columns, values, size(prepared.R_force)...)
end
RForceCollocated = collocated_source_operator(preparedOPT, sourceNodesOPT)
preparedOPTWithoutGamma = runOPTWithoutGamma ?
    flexOPT._prepared_with_matrices(preparedOPT, preparedOPT.A_unknown,
        preparedOPT.L_known, RForceCollocated; source_mode=:collocated) : nothing
propagationOPTWithoutGamma = runOPTWithoutGamma ? propagateLinearSystem(
    preparedOPTWithoutGamma, optSteps, dtOPT; sourceFull=sourceOPT,
    output_stride=optOutputStride, blowup_limit=1e8,
    solver_name="OPT3 without Γ redistribution",
) : nothing
runOPTWithoutGamma && @assert !propagationOPTWithoutGamma.stopped_early
uxOPT = propagationOPT.history[:, :, 1, :]
uzOPT = propagationOPT.history[:, :, 2, :]
optTimes = propagationOPT.times
uxOPTWithoutGamma = runOPTWithoutGamma ?
    propagationOPTWithoutGamma.history[:, :, 1, :] : nothing
uzOPTWithoutGamma = runOPTWithoutGamma ?
    propagationOPTWithoutGamma.history[:, :, 2, :] : nothing

# Higher-order runs are gated by ElasticLHSOperatorAudit.ipynb.
if runOPT4
    sourceOPT4 = zeros(Float64, preparedOPT4.NforcePoints,
        preparedOPT4.NForceField, length(sourceTimesOPT))
    for sourceNode in sourceNodesOPT
        sourceOPT4[sourceNode.linear, 2, :] .+=
            sourceNode.weight .* sourceScaleOPT .* waveletOPT
    end
    propagationOPT4 = propagateLinearSystem(preparedOPT4, optSteps, dtOPT;
        sourceFull=sourceOPT4, output_stride=optOutputStride,
        blowup_limit=1e8, solver_name="OPT4")
    @assert !propagationOPT4.stopped_early
    uxOPT4 = propagationOPT4.history[:, :, 1, :]
    uzOPT4 = propagationOPT4.history[:, :, 2, :]
    opt4Times = propagationOPT4.times
else
    propagationOPT4 = uxOPT4 = uzOPT4 = opt4Times = nothing
end

# OPT5 remains available, but is skipped in the quick benchmark by default.
if runOPT5
    sourceOPT5 = zeros(Float64, preparedOPT5.NforcePoints,
        preparedOPT5.NForceField, length(sourceTimesOPT))
    for sourceNode in sourceNodesOPT
        sourceOPT5[sourceNode.linear, 2, :] .+=
            sourceNode.weight .* sourceScaleOPT .* waveletOPT
    end
    propagationOPT5 = propagateLinearSystem(
        preparedOPT5, optSteps, dtOPT;
        sourceFull=sourceOPT5, output_stride=optOutputStride,
        blowup_limit=1e8, solver_name="OPT5",
    )
    @assert !propagationOPT5.stopped_early
    uxOPT5 = propagationOPT5.history[:, :, 1, :]
    uzOPT5 = propagationOPT5.history[:, :, 2, :]
    opt5Times = propagationOPT5.times
else
    propagationOPT5 = uxOPT5 = uzOPT5 = opt5Times = nothing
end

# Run the less stable delta-test conventional recipe last, so an eventual
# late stop cannot discard the expensive OPT5 result from this session.
sourceConvFD = zeros(Float64, preparedConvFD.NforcePoints,
    preparedConvFD.NForceField, length(sourceTimesOPT))
for sourceNode in sourceNodesOPT
    sourceConvFD[sourceNode.linear, 2, :] .+=
        sourceNode.weight .* sourceScaleOPT .* waveletOPT
end
# orderB=-1 is an interior conventional stencil, not a stable closure for
# the abruptly truncated outer rows. Stop before the earliest source tail
# can excite those rows unless the user explicitly requests the diagnostic.
convFDBoundaryGuard = 0.5 / sourceFrequency
convFDBoundarySafeTime = max(0.0, earliestPEdgeArrival - convFDBoundaryGuard)
convFDDuration = runConvFDBeyondBoundarySafeTime ? duration :
    min(duration, convFDBoundarySafeTime)
convFDSteps = ceil(Int, convFDDuration / dtOPT)
propagationConvFD = propagateLinearSystem(
    preparedConvFD, convFDSteps, dtOPT;
    sourceFull=sourceConvFD, output_stride=optOutputStride,
    blowup_limit=1e8,
    solver_name="convFD3 (orderB=-1)",
)
uxConvFD = propagationConvFD.history[:, :, 1, :]
uzConvFD = propagationConvFD.history[:, :, 2, :]
convFDTimes = propagationConvFD.times
propagationConvFD.stopped_early && @warn(
    "convFD3 stopped early; its partial history remains available",
    final_time=last(convFDTimes), requested_duration=duration)
@show convFDBoundarySafeTime convFDDuration propagationConvFD.stopped_early
@show dtOPT size(uzOPT) maximum(abs, uxOPT) maximum(abs, uzOPT)
@show size(uzConvFD) maximum(abs, uxConvFD) maximum(abs, uzConvFD)
runOPT5 && @show size(uzOPT5) maximum(abs, uxOPT5) maximum(abs, uzOPT5)


## Common source-time function

In [ ]:
# This is the actual Δt written to SPECFEM's Par_file below. It is not
# a waveform-alignment parameter; all three traces retain physical time.
dtSPECFEM = dtOPT # exact common physical time step for OPT and SPECFEM
sourceTimesFD = (1:fdSteps) .* Float64(fd.dt)
sourceTimesSPECFEM = (0:ceil(Int, duration / dtSPECFEM)-1) .*
    dtSPECFEM
sourceFigure = Figure(size=(1050, 720))
waveletAxis = Axis(sourceFigure[1, 1]; xlabel="time (s)",
    ylabel="normalized Ricker",
    title="Same temporal force function, sampled by each solver")
forceAxis = Axis(sourceFigure[2, 1]; xlabel="time (s)",
    ylabel="vertical line force (N/m)",
    title="Physical source amplitude")
lines!(waveletAxis, sourceTimesFD, rickerSource.(sourceTimesFD);
    label="FD3, Δt=$(round(Float64(fd.dt); sigdigits=4)) s")
lines!(waveletAxis, sourceTimesOPT, rickerSource.(sourceTimesOPT);
    label="OPT3, Δt=$(round(dtOPT; sigdigits=4)) s")
lines!(waveletAxis, sourceTimesSPECFEM, rickerSource.(sourceTimesSPECFEM);
    label="SPECFEM2D, Δt=$(round(dtSPECFEM; sigdigits=4)) s",
    linestyle=:dash)
lines!(forceAxis, sourceTimesFD, sourceForce .* rickerSource.(sourceTimesFD);
    label="common F_z(t)", color=:black)
axislegend(waveletAxis; position=:rb)
axislegend(forceAxis; position=:rb)
display(sourceFigure)
nothing


## OPT Γ source redistribution

In [ ]:
# A unit value at one f_z source point is mapped through the assembled
# Γ/R_force operator onto neighboring residual test functions.
gammaProbe = zeros(Float64, preparedOPT.NforcePoints,
    preparedOPT.NForceField, preparedOPT.timePointsUsedForOneStep)
gammaProbe[sourceLinearOPT, 2, :] .= 1.0
gammaResidual = preparedOPT.R_force * vec(gammaProbe)
gammaResidualByField = reshape(gammaResidual,
    preparedOPT.NField, preparedOPT.NpointsSpace)
gammaVertical = reshape(gammaResidualByField[2, :], preparedOPT.spaceShape)
gammaTolerance = maximum(abs, gammaVertical) * 1e-12
gammaSupport = findall(abs.(gammaVertical) .> gammaTolerance)
gammaDiagnostic = (
    input_point=sourceIndexOPT,
    redistributed_points=length(gammaSupport),
    coefficient_sum=sum(gammaVertical),
    coefficient_l1=sum(abs, gammaVertical),
    coefficient_l2=norm(gammaVertical),
)
function sparse_row_summary(matrix, row)
    columns, coefficients = findnz(vec(matrix[row, :]))
    (nnz=length(coefficients), sum=sum(coefficients),
        l1=sum(abs, coefficients), l2=norm(coefficients),
        maximum=isempty(coefficients) ? 0.0 : maximum(abs, coefficients),
        columns=columns, coefficients=coefficients)
end
sourceResidualRow = 2 + preparedOPT.NField*(sourceLinearOPT-1)
localOperatorSummary = (
    lhs_future=sparse_row_summary(preparedOPT.A_unknown, sourceResidualRow),
    lhs_past_present=sparse_row_summary(preparedOPT.L_known, sourceResidualRow),
    rhs_Gamma=sparse_row_summary(preparedOPT.R_force, sourceResidualRow),
    rhs_collocated=sparse_row_summary(RForceCollocated, sourceResidualRow),
)
@show gammaDiagnostic
display(localOperatorSummary)
gammaFigure = Figure(size=(760, 620))
gammaAxis = Axis(gammaFigure[1, 1]; xlabel="x (km)", ylabel="z (km)",
    title="OPT Γ/R_force response to one vertical source point",
    aspect=DataAspect())
gammaPlot = heatmap!(gammaAxis, collect(xOPT) ./ 1e3, collect(zOPT) ./ 1e3,
    gammaVertical; colormap=:balance)
Colorbar(gammaFigure[1, 2], gammaPlot; label="Γ coefficient")
xlims!(gammaAxis, sourcePosition.x / 1e3 - 5, sourcePosition.x / 1e3 + 5)
ylims!(gammaAxis, sourcePosition.z / 1e3 - 5, sourcePosition.z / 1e3 + 5)
display(gammaFigure)
nothing


## Interior receiver velocity traces (`uₓ`, `u_z`)

In [ ]:
waveformQuantity = @isdefined(waveformQuantity) ? waveformQuantity : :velocity
function displacement_traces(history, times, xaxis, zindex, receivers)
    traces = Matrix{Float64}(undef, length(times), length(receivers))
    for (j, receiver) in enumerate(receivers)
        ix = argmin(abs.(xaxis .- receiver))
        traces[:, j] .= Float64.(history[ix, zindex, :])
    end
    return (time=Float64.(times), values=traces)
end

function velocity_traces(history, times, xaxis, zindex, receivers)
    traces = Matrix{Float64}(undef, length(times) - 1, length(receivers))
    for (j, receiver) in enumerate(receivers)
        ix = argmin(abs.(xaxis .- receiver))
        traces[:, j] .= diff(Float64.(history[ix, zindex, :])) ./ diff(times)
    end
    return (time=(times[1:end-1] .+ times[2:end]) ./ 2, values=traces)
end
function velocity_traces_at_points(history, times, xaxis, zaxis, receivers;
    sampling=:nearest)
    sampling === :nearest || error("only :nearest is enabled in this cheap benchmark path")
    traces = Matrix{Float64}(undef, length(times) - 1, length(receivers))
    for (j, receiver) in enumerate(receivers)
        ix = argmin(abs.(xaxis .- receiver.x))
        iz = argmin(abs.(zaxis .- receiver.z))
        traces[:, j] .= diff(Float64.(history[ix, iz, :])) ./ diff(times)
    end
    (time=(times[1:end-1] .+ times[2:end]) ./ 2, values=traces,
        receivers=collect(receivers), sampling)
end
fdReceiverZIndex = argmin(abs.(fdCoordinates.z .- receiverZ))
optReceiverZIndex = argmin(abs.(zOPT .- receiverZ))
fdDisplacementX = displacement_traces(uxFD, fdTimes, fdCoordinates.x,
    fdReceiverZIndex, receiverX)
fdDisplacementZ = displacement_traces(uzFD, fdTimes, fdCoordinates.x,
    fdReceiverZIndex, receiverX)
optDisplacementX = displacement_traces(uxOPT, optTimes, xOPT,
    optReceiverZIndex, receiverX)
optDisplacementZ = displacement_traces(uzOPT, optTimes, xOPT,
    optReceiverZIndex, receiverX)
convFDDisplacementX = displacement_traces(uxConvFD, convFDTimes, xOPT,
    optReceiverZIndex, receiverX)
convFDDisplacementZ = displacement_traces(uzConvFD, convFDTimes, xOPT,
    optReceiverZIndex, receiverX)
optNoGammaDisplacementX = runOPTWithoutGamma ? displacement_traces(
    uxOPTWithoutGamma, optTimes, xOPT, optReceiverZIndex, receiverX) : nothing
optNoGammaDisplacementZ = runOPTWithoutGamma ? displacement_traces(
    uzOPTWithoutGamma, optTimes, xOPT, optReceiverZIndex, receiverX) : nothing
opt4DisplacementX = runOPT4 ? displacement_traces(
    uxOPT4, opt4Times, xOPT, optReceiverZIndex, receiverX) : nothing
opt4DisplacementZ = runOPT4 ? displacement_traces(
    uzOPT4, opt4Times, xOPT, optReceiverZIndex, receiverX) : nothing
opt5DisplacementX = runOPT5 ? displacement_traces(
    uxOPT5, opt5Times, xOPT, optReceiverZIndex, receiverX) : nothing
opt5DisplacementZ = runOPT5 ? displacement_traces(
    uzOPT5, opt5Times, xOPT, optReceiverZIndex, receiverX) : nothing

fdVelocityX = velocity_traces(uxFD, fdTimes, fdCoordinates.x,
    fdReceiverZIndex, receiverX)
fdVelocityZ = velocity_traces(uzFD, fdTimes, fdCoordinates.x,
    fdReceiverZIndex, receiverX)
optVelocityX = velocity_traces(uxOPT, optTimes, xOPT,
    optReceiverZIndex, receiverX)
optVelocityZ = velocity_traces(uzOPT, optTimes, xOPT,
    optReceiverZIndex, receiverX)
convFDVelocityX = velocity_traces(uxConvFD, convFDTimes, xOPT,
    optReceiverZIndex, receiverX)
convFDVelocityZ = velocity_traces(uzConvFD, convFDTimes, xOPT,
    optReceiverZIndex, receiverX)
optNoGammaVelocityX = runOPTWithoutGamma ? velocity_traces(
    uxOPTWithoutGamma, optTimes, xOPT, optReceiverZIndex, receiverX) : nothing
optNoGammaVelocityZ = runOPTWithoutGamma ? velocity_traces(
    uzOPTWithoutGamma, optTimes, xOPT, optReceiverZIndex, receiverX) : nothing
opt4VelocityX = runOPT4 ? velocity_traces(uxOPT4, opt4Times, xOPT,
    optReceiverZIndex, receiverX) : nothing
opt4VelocityZ = runOPT4 ? velocity_traces(uzOPT4, opt4Times, xOPT,
    optReceiverZIndex, receiverX) : nothing
opt5VelocityX = runOPT5 ? velocity_traces(uxOPT5, opt5Times, xOPT,
    optReceiverZIndex, receiverX) : nothing
opt5VelocityZ = runOPT5 ? velocity_traces(uzOPT5, opt5Times, xOPT,
    optReceiverZIndex, receiverX) : nothing
greenSourceReferences = map(sourceSpecsReference) do sourceSpec
    aki_richards_line_force_2d(
        fdTimes, [(x=xr, z=receiverZ) for xr in receiverX];
        source=(x=sourceSpec.x, z=sourceSpec.z),
        vp=vp0, vs=vs0, rho=rho0,
        force_time_function=rickerSource,
        force_amplitude=sourceForce * sourceSpec.weight,
        source_frequency=sourceFrequency, out_of_plane_step=250.0,
    )
end
greenReference = merge(first(greenSourceReferences), (
    displacement=sum(ref.displacement for ref in greenSourceReferences),
    velocity=sum(ref.velocity for ref in greenSourceReferences),
))
greenVelocityX = (time=greenReference.velocity_time,
    values=greenReference.velocity[:, :, 1])
greenVelocityZ = (time=greenReference.velocity_time,
    values=greenReference.velocity[:, :, 2])
greenDisplacementX = (time=greenReference.time,
    values=greenReference.displacement[:, :, 1])
greenDisplacementZ = (time=greenReference.time,
    values=greenReference.displacement[:, :, 2])

if waveformQuantity === :velocity
    fdWaveformX, fdWaveformZ = fdVelocityX, fdVelocityZ
    optWaveformX, optWaveformZ = optVelocityX, optVelocityZ
    convFDWaveformX, convFDWaveformZ = convFDVelocityX, convFDVelocityZ
    optNoGammaWaveformX, optNoGammaWaveformZ =
        optNoGammaVelocityX, optNoGammaVelocityZ
    opt4WaveformX, opt4WaveformZ = opt4VelocityX, opt4VelocityZ
    opt5WaveformX, opt5WaveformZ = opt5VelocityX, opt5VelocityZ
    greenWaveformX, greenWaveformZ = greenVelocityX, greenVelocityZ
    waveformUnitLabel, waveformComponentPrefix = "m/s", "v"
else
    fdWaveformX, fdWaveformZ = fdDisplacementX, fdDisplacementZ
    optWaveformX, optWaveformZ = optDisplacementX, optDisplacementZ
    convFDWaveformX, convFDWaveformZ = convFDDisplacementX, convFDDisplacementZ
    optNoGammaWaveformX, optNoGammaWaveformZ =
        optNoGammaDisplacementX, optNoGammaDisplacementZ
    opt4WaveformX, opt4WaveformZ = opt4DisplacementX, opt4DisplacementZ
    opt5WaveformX, opt5WaveformZ = opt5DisplacementX, opt5DisplacementZ
    greenWaveformX, greenWaveformZ = greenDisplacementX, greenDisplacementZ
    waveformUnitLabel, waveformComponentPrefix = "m", "u"
end
fdVelocity, optVelocity = fdVelocityZ, optVelocityZ # physical velocity aliases
receiver = cld(length(receiverX), 2)
fdTrace = (time=fdWaveformZ.time, values=fdWaveformZ.values[:, receiver])
optTrace = (time=optWaveformZ.time, values=optWaveformZ.values[:, receiver])
convFDTrace = (time=convFDWaveformZ.time,
    values=convFDWaveformZ.values[:, receiver])
optNoGammaTrace = runOPTWithoutGamma ? (time=optNoGammaWaveformZ.time,
    values=optNoGammaWaveformZ.values[:, receiver]) : nothing
opt4Trace = runOPT4 ? (time=opt4WaveformZ.time,
    values=opt4WaveformZ.values[:, receiver]) : nothing
opt5Trace = runOPT5 ? (time=opt5WaveformZ.time,
    values=opt5WaveformZ.values[:, receiver]) : nothing
greenTrace = (time=greenWaveformZ.time,
    values=greenWaveformZ.values[:, receiver])
preSPECFEMTraces = (Green2D=greenTrace, explicitFD3=fdTrace,
    convFD3=convFDTrace, OPT3=optTrace)
runOPTWithoutGamma && (preSPECFEMTraces = merge(preSPECFEMTraces,
    (OPT3_no_Gamma=optNoGammaTrace,)))
runOPT4 && (preSPECFEMTraces = merge(preSPECFEMTraces, (OPT4=opt4Trace,)))
runOPT5 && (preSPECFEMTraces = merge(preSPECFEMTraces, (OPT5=opt5Trace,)))
preSPECFEM = plot_solver_benchmark(preSPECFEMTraces;
    title="Homogeneous interior model — normalized $(waveformComponentPrefix)_z")
display(preSPECFEM.figure)
nothing


## SPECFEM2D reference

In [ ]:
caseDirectory = joinpath(flexopt_root, "data", "specfem2d_benchmarks",
    "homogeneous_flat")
solidZ = applyFreeSurface ? findall(z .<= 0.0) : collect(eachindex(z))
zSPECFEM = z[solidZ]
vp = fill(vp0, nx, length(solidZ))
vs = fill(vs0, nx, length(solidZ))
rho = fill(rho0, nx, length(solidZ))
# Refining FD/OPT must not silently refine the SPECFEM spectral mesh.
specfemNxElements = round(Int, (last(x)-first(x)) / (4referenceDx))
specfemNzElements = round(Int, (last(zSPECFEM)-first(zSPECFEM)) / (4referenceDx))
specfemCase = prepare_specfem2d_case(
    caseDirectory, x, zSPECFEM, vp, vs, rho, fill(last(zSPECFEM), nx);
    source=sourcePosition, sources=sourceSpecsReference, receivers=receiverX,
    duration=duration, dt=dtSPECFEM, f0=sourceFrequency,
    source_factor=sourceForce, source_angle=0.0,
    source_time_function=rickerSource,
    nx_elements=specfemNxElements, nz_elements=specfemNzElements,
    receiver_z=receiverZ, free_surface=applyFreeSurface,
    snapshot_interval_steps=max(1, round(Int, outputSampling / dtSPECFEM)),
    snapshot_image_type=5, # vertical velocity, matching the seismograms
    output_wavefield_dumps=true,
    wavefield_dump_type=1, # displacement vector (uₓ, u_z)
    binary_wavefield_dumps=true,
)
if runSPECFEM2D
    specfemRun = run_specfem2d_case(specfemCase.case_directory)
else
    specfemRun = (output=specfemCase.output,)
end
verticalFiles = find_specfem2d_traces(specfemRun.output; component=:z)
@assert length(verticalFiles) == length(receiverX)
specfemTracesZ = [read_specfem2d_trace(file;
    time_shift=specfemCase.time_axis_shift) for file in verticalFiles]
horizontalFiles = find_specfem2d_traces(specfemRun.output; component=:x)
@assert length(horizontalFiles) == length(receiverX)
specfemTracesX = [read_specfem2d_trace(file;
    time_shift=specfemCase.time_axis_shift) for file in horizontalFiles]
function integrate_velocity_trace(trace)
    displacement = zeros(Float64, length(trace.values))
    displacement[2:end] .= cumsum(
        ((trace.values[1:end-1] .+ trace.values[2:end]) ./ 2) .*
        diff(trace.time))
    (time=Float64.(trace.time), values=displacement)
end
specfemWaveformsX = waveformQuantity === :velocity ? specfemTracesX :
    integrate_velocity_trace.(specfemTracesX)
specfemWaveformsZ = waveformQuantity === :velocity ? specfemTracesZ :
    integrate_velocity_trace.(specfemTracesZ)
specfemTraces = specfemWaveformsZ # selected-quantity compatibility alias
specfemTrace = specfemWaveformsZ[receiver]
specfemWavefield = read_specfem2d_wavefield_dumps(
    specfemRun.output; dt=dtSPECFEM,
    time_axis_shift=specfemCase.time_axis_shift,
)
specfemSnapshotVideo = make_specfem2d_snapshot_video(
    specfemRun.output;
    output_path=joinpath(specfemRun.output, "SPECFEM2D_vertical_velocity.mp4"),
    framerate=20,
)
solverLogText = read(joinpath(specfemCase.case_directory, "solver.log"), String)
specfemCFLMatch = match(r"Max CFL stability condition[^=]*=\s*([0-9.Ee+-]+)",
    solverLogText)
specfemCFL = isnothing(specfemCFLMatch) ? missing :
    parse(Float64, specfemCFLMatch.captures[1])
cflSummary = (
    FD3_axis=vp0 * Float64(fd.dt) / dx,
    FD3_2D=sqrt(2) * vp0 * Float64(fd.dt) / dx,
    OPT3_axis=vp0 * dtOPT / dxOPT,
    OPT3_2D=sqrt(2) * vp0 * dtOPT / dxOPT,
    SPECFEM2D_reported=specfemCFL,
)
# SPECFEM uses 5 GLL points per element in this build. Its default number
# of elements makes the mean GLL-node interval comparable to dx.
specfemElementSize = (
    x=(last(x)-first(x)) / specfemCase.nx_elements,
    z=(last(zSPECFEM)-first(zSPECFEM)) / specfemCase.nz_elements,
)
discretizationSummary = (
    OPT3=(dx=dxOPT, dt=dtOPT),
    SPECFEM2D=(element_size=specfemElementSize,
        mean_GLL_interval=(x=specfemElementSize.x/4,
                           z=specfemElementSize.z/4), dt=dtSPECFEM),
)
@show cflSummary discretizationSummary specfemCase.time_axis_shift
@show specfemCase.case_directory dtSPECFEM specfemSnapshotVideo.path
if endswith(lowercase(specfemSnapshotVideo.path), ".gif")
    gifData = base64encode(read(specfemSnapshotVideo.path))
    display(MIME("text/html"),
        "<img src='data:image/gif;base64,$(gifData)' style='max-width:100%'>")
else
    # VS Code/IJulia renders the MP4 with native playback controls.
    videoURL = "file://" * specfemSnapshotVideo.path
    display(MIME("text/html"),
        "<video controls loop src='$(videoURL)' style='max-width:100%'></video>")
end
specfemSnapshotVideo.path


## Shape and raw-amplitude comparison

In [ ]:
traces = (Green2D=greenTrace, explicitFD3=fdTrace,
    convFD3=convFDTrace, OPT3=optTrace, SPECFEM2D=specfemTrace)
runOPT4 && (traces = merge(traces, (OPT4=opt4Trace,)))
runOPT5 && (traces = merge(traces, (OPT5=opt5Trace,)))
# 501 common samples are ample at this source bandwidth and keep this
# diagnostic cell interactive; the solvers themselves are not resampled.
metricTraces = (explicitFD3=fdTrace, convFD3=convFDTrace,
    OPT3=optTrace, SPECFEM2D=specfemTrace)
runOPT4 && (metricTraces = merge(metricTraces, (OPT4=opt4Trace,)))
runOPT5 && (metricTraces = merge(metricTraces, (OPT5=opt5Trace,)))
metricsToGreen = map(trace -> waveform_metrics(
    greenTrace, trace; samples=501), metricTraces)
solverTracesForSPECFEM = (convFD3=convFDTrace, OPT3=optTrace)
runOPT4 && (solverTracesForSPECFEM = merge(solverTracesForSPECFEM,
    (OPT4=opt4Trace,)))
runOPT5 && (solverTracesForSPECFEM = merge(solverTracesForSPECFEM,
    (OPT5=opt5Trace,)))
metricsToSPECFEM = map(trace -> waveform_metrics(
    specfemTrace, trace; samples=501), solverTracesForSPECFEM)
peakVelocity = map(trace -> maximum(abs, trace.values), traces)
peakRatioToGreen = map(value -> value / peakVelocity.Green2D, peakVelocity)
@show metricsToGreen
@show metricsToSPECFEM
@show peakVelocity peakRatioToGreen
normalizedPlot = plot_solver_benchmark(traces;
    title="Homogeneous flat model — waveform shape")
rawPlot = plot_solver_benchmark(traces; normalize=false,
    title="Homogeneous flat model — raw vertical $(waveformQuantity)")
display(normalizedPlot.figure)
display(rawPlot.figure)
nothing


## `vₓ` and `v_z` velocity errors against Green at every interior station

In [ ]:
function artificial_boundary_windows(
    receiverX;
    source=sourcePosition,
    xmin=first(x), xmax=last(x), zmin=first(z), zmax=last(z),
    has_free_surface=applyFreeSurface,
    receiver_z=receiverZ, velocity=vp0, source_delay=sourceDelay,
    source_frequency=sourceFrequency, duration=duration,
)
    # Image sources give the first geometrically possible P reflection from
    # each artificial boundary. The free surface z=0 is deliberately absent.
    sideAndBottomImages = (
        left=(x=2xmin - source.x, z=source.z),
        right=(x=2xmax - source.x, z=source.z),
        bottom=(x=source.x, z=2zmin - source.z),
    )
    images = has_free_surface ? sideAndBottomImages : merge(
        sideAndBottomImages,
        (top=(x=source.x, z=2zmax - source.z),),
    )
    map(receiverX) do xr
        directDistance = hypot(xr - source.x, receiver_z - source.z)
        reflectedDistances = map(image ->
            hypot(xr - image.x, receiver_z - image.z), images)
        directArrival = source_delay + directDistance / velocity
        directSArrival = source_delay + directDistance / vs0
        artificialArrival = source_delay + minimum(reflectedDistances) / velocity
        # One half-period on each side avoids comparing only a clipped peak.
        start = max(0.0, directArrival - 0.5 / source_frequency)
        stop = min(duration, artificialArrival - 0.5 / source_frequency)
        stop > start || error("No uncontaminated time window at x=$xr")
        (start=start, stop=stop, direct_arrival=directArrival,
         direct_s_arrival=directSArrival,
         first_artificial_arrival=artificialArrival)
    end
end
safeWindows = artificial_boundary_windows(receiverX)
@assert all(window -> window.direct_s_arrival <= window.stop, safeWindows)
@show safeWindows

function linear_trace_sample(trace, query_times)
    map(query_times) do time
        time <= first(trace.time) && return first(trace.values)
        time >= last(trace.time) && return last(trace.values)
        left = searchsortedlast(trace.time, time)
        α = (time - trace.time[left]) /
            (trace.time[left + 1] - trace.time[left])
        (1 - α) * trace.values[left] + α * trace.values[left + 1]
    end
end

function plot_station_waveform_errors(
    receiverX, greenWaveform, convFDWaveform, optWaveform,
    opt4Waveform, opt5Waveform, specfemWaveforms;
    selected=eachindex(receiverX),
    safe_windows=nothing,
    component_label="v_z", unit_label="m/s",
)
    selected = collect(selected)
    figure = Figure(size=(1250, 245 * length(selected)))
    colors = Makie.wong_colors()
    for (row, station) in enumerate(selected)
        stationLabel = "x = $(receiverX[station] / 1e3) km"
        velocityAxis = Axis(
            figure[row, 1];
            xlabel=row == length(selected) ? "time (s)" : "",
            ylabel="$component_label ($unit_label)",
            title="$stationLabel — $(waveformQuantity)",
        )
        residualAxis = Axis(
            figure[row, 2];
            xlabel=row == length(selected) ? "time (s)" : "",
            ylabel="method − Green ($unit_label)",
            title="$stationLabel — $(waveformQuantity) error",
        )
        if !isnothing(safe_windows)
            window = safe_windows[station]
            vspan!(velocityAxis, window.start, window.stop;
                color=(:seagreen, 0.10))
            vspan!(residualAxis, window.start, window.stop;
                color=(:seagreen, 0.10))
            vlines!(velocityAxis, [window.stop]; color=:seagreen,
                linestyle=:dash, linewidth=1.5)
            vlines!(residualAxis, [window.stop]; color=:seagreen,
                linestyle=:dash, linewidth=1.5)
            vlines!(velocityAxis, [window.direct_arrival]; color=:dodgerblue,
                linestyle=:dot, linewidth=1.5)
            vlines!(residualAxis, [window.direct_arrival]; color=:dodgerblue,
                linestyle=:dot, linewidth=1.5)
            if window.direct_s_arrival <= window.stop
                vlines!(velocityAxis, [window.direct_s_arrival]; color=:darkorange,
                    linestyle=:dot, linewidth=1.5)
                vlines!(residualAxis, [window.direct_s_arrival]; color=:darkorange,
                    linestyle=:dot, linewidth=1.5)
            end
        end
        greenTrace = (time=greenWaveform.time,
            values=greenWaveform.values[:, station])
        stationTraces = (
            convFD3=(time=convFDWaveform.time,
                values=convFDWaveform.values[:, station]),
            OPT3=(time=optWaveform.time, values=optWaveform.values[:, station]),
            SPECFEM2D=specfemWaveforms[station],
        )
        !isnothing(opt4Waveform) && (stationTraces = merge(
            stationTraces, (OPT4=(time=opt4Waveform.time,
                values=opt4Waveform.values[:, station]),)))
        !isnothing(opt5Waveform) && (stationTraces = merge(
            stationTraces, (OPT5=(time=opt5Waveform.time,
                values=opt5Waveform.values[:, station]),)))
        lines!(velocityAxis, greenTrace.time, greenTrace.values;
            label="Green2D", color=:black, linewidth=2)
        window = isnothing(safe_windows) ?
            (start=max(first(greenTrace.time), maximum(first(t.time) for t in values(stationTraces))),
             stop=min(last(greenTrace.time), minimum(last(t.time) for t in values(stationTraces)))) :
            safe_windows[station]
        residualTimes = greenTrace.time[(greenTrace.time .>= window.start) .&
            (greenTrace.time .<= window.stop)]
        greenOnResidualTimes = linear_trace_sample(greenTrace, residualTimes)
        for (solverIndex, (solver, trace)) in enumerate(pairs(stationTraces))
            color = colors[solverIndex]
            methodOnResidualTimes = linear_trace_sample(trace, residualTimes)
            residual = methodOnResidualTimes .- greenOnResidualTimes
            metrics = waveform_metrics(
                (time=residualTimes, values=greenOnResidualTimes),
                (time=residualTimes, values=methodOnResidualTimes);
                samples=min(501, length(residualTimes)))
            label = "$(solver), relRMSE=$(round(metrics.relative_rmse; sigdigits=3))"
            lines!(velocityAxis, trace.time, trace.values; label, color)
            lines!(residualAxis, residualTimes, residual; label, color)
        end
        hlines!(residualAxis, [0.0]; color=(:black, 0.5), linewidth=1)
        row == 1 && axislegend(velocityAxis; position=:rb)
        row == 1 && axislegend(residualAxis; position=:rb)
    end
    linkxaxes!(filter(!isnothing, [content(figure[r, c])
        for r in eachindex(selected), c in 1:2])...)
    return figure
end

# `waveformQuantity` is an actual data selector, not metadata.
selectedStations = collect(eachindex(receiverX))
waveformQuantityCheck = waveformQuantity === :velocity ? (
    Green2D=:analytical_velocity,
    convFD3=:time_derivative_of_displacement,
    OPT3=:time_derivative_of_displacement,
    SPECFEM2D=:seismotype_2_semv_velocity,
    units=:metres_per_second,
) : (
    Green2D=:analytical_displacement,
    convFD3=:direct_displacement_history,
    OPT3=:direct_displacement_history,
    SPECFEM2D=:time_integral_of_semv_velocity,
    units=:metres,
)
runOPT5 && (waveformQuantityCheck = merge(waveformQuantityCheck,
    (OPT5=waveformQuantity === :velocity ?
        :time_derivative_of_displacement : :direct_displacement_history,)))
runOPT4 && (waveformQuantityCheck = merge(waveformQuantityCheck,
    (OPT4=waveformQuantity === :velocity ?
        :time_derivative_of_displacement : :direct_displacement_history,)))
@show waveformQuantity waveformQuantityCheck
stationWaveformFigureX = plot_station_waveform_errors(
    receiverX, greenWaveformX, convFDWaveformX,
    optWaveformX, opt4WaveformX, opt5WaveformX, specfemWaveformsX;
    selected=selectedStations,
    safe_windows=safeWindows,
    component_label="$(waveformComponentPrefix)ₓ",
    unit_label=waveformUnitLabel,
)
stationWaveformFigureZ = plot_station_waveform_errors(
    receiverX, greenWaveformZ, convFDWaveformZ,
    optWaveformZ, opt4WaveformZ, opt5WaveformZ, specfemWaveformsZ;
    selected=selectedStations,
    safe_windows=safeWindows,
    component_label="$(waveformComponentPrefix)_z",
    unit_label=waveformUnitLabel,
)
display(stationWaveformFigureX)
display(stationWaveformFigureZ)
# SPECFEM-referenced errors at every station, on the uncontaminated window.
cropForSPECFEMMetric(trace, window) = begin
    keep = (trace.time .>= window.start) .& (trace.time .<= window.stop)
    (time=trace.time[keep], values=trace.values[keep])
end
metricsAgainstSPECFEM = map(eachindex(receiverX)) do station
    reference = specfemWaveformsZ[station]
    candidates = (convFD3=(time=convFDWaveformZ.time,
            values=convFDWaveformZ.values[:, station]),
        OPT3=(time=optWaveformZ.time, values=optWaveformZ.values[:, station]))
    runOPT4 && (candidates = merge(candidates,
        (OPT4=(time=opt4WaveformZ.time,
            values=opt4WaveformZ.values[:, station]),)))
    runOPT5 && (candidates = merge(candidates,
        (OPT5=(time=opt5WaveformZ.time,
            values=opt5WaveformZ.values[:, station]),)))
    window = safeWindows[station]
    croppedReference = cropForSPECFEMMetric(reference, window)
    metrics = map(trace -> waveform_metrics(croppedReference,
        cropForSPECFEMMetric(trace, window); samples=501), candidates)
    (x_km=receiverX[station] / 1e3, metrics...)
end
display(metricsAgainstSPECFEM)
nothing


In [ ]:
# Estimate phase delay without modifying or realigning the displayed traces.
# Positive lag means that the numerical method arrives later than Green2D.
function best_trace_lag(reference, candidate, window;
    max_lag=0.5 / sourceFrequency, lag_samples=201, samples=501)
    start = max(window.start, first(reference.time),
        first(candidate.time) + max_lag)
    stop = min(window.stop, last(reference.time),
        last(candidate.time) - max_lag)
    stop > start || error("No common interval remains for lag estimation")
    times = collect(range(start, stop; length=samples))
    reference_values = linear_trace_sample(reference, times)
    reference_centered = reference_values .- mean(reference_values)
    reference_norm = max(norm(reference_values), eps(Float64))
    lags = range(-max_lag, max_lag; length=lag_samples)
    correlations = map(lags) do lag
        candidate_values = linear_trace_sample(candidate, times .+ lag)
        candidate_centered = candidate_values .- mean(candidate_values)
        dot(reference_centered, candidate_centered) /
            max(norm(reference_centered) * norm(candidate_centered), eps(Float64))
    end
    best_index = argmax(correlations)
    best_lag = Float64(lags[best_index])
    raw_values = linear_trace_sample(candidate, times)
    aligned_values = linear_trace_sample(candidate, times .+ best_lag)
    return (
        lag_s=best_lag,
        correlation=correlations[best_index],
        raw_relative_rmse=norm(raw_values .- reference_values) / reference_norm,
        shifted_relative_rmse=norm(aligned_values .- reference_values) / reference_norm,
    )
end

timeShiftDiagnostics = map(eachindex(receiverX)) do station
    window = safeWindows[station]
    reference = (time=greenWaveformZ.time,
        values=greenWaveformZ.values[:, station])
    methods = (
        convFD3=(time=convFDWaveformZ.time,
            values=convFDWaveformZ.values[:, station]),
        OPT3=(time=optWaveformZ.time,
            values=optWaveformZ.values[:, station]),
        SPECFEM2D=specfemWaveformsZ[station],
    )
    runOPT4 && (methods = merge(methods, (OPT4=(time=opt4WaveformZ.time,
        values=opt4WaveformZ.values[:, station]),)))
    runOPT5 && (methods = merge(methods, (OPT5=(time=opt5WaveformZ.time,
        values=opt5WaveformZ.values[:, station]),)))
    metrics = map(trace -> best_trace_lag(reference, trace, window), methods)
    (x_km=receiverX[station] / 1e3, metrics...)
end
foreach(display, timeShiftDiagnostics)

lagFigure = Figure(size=(820, 480))
lagAxis = Axis(lagFigure[1, 1]; xlabel="receiver x (km)",
    ylabel="best lag relative to Green2D (s)",
    title="Positive lag = numerical waveform arrives later")
lagMethods = Tuple(vcat([:convFD3, :OPT3],
    runOPT4 ? [:OPT4] : Symbol[],
    runOPT5 ? [:OPT5] : Symbol[],
    [:SPECFEM2D]))
for method in lagMethods
    scatterlines!(lagAxis, getproperty.(timeShiftDiagnostics, :x_km),
        [getproperty(getproperty(item, method), :lag_s)
            for item in timeShiftDiagnostics]; marker=:circle, label=string(method))
end
hlines!(lagAxis, [0.0]; color=:black, linestyle=:dash)
axislegend(lagAxis; position=:rb)
display(lagFigure)
nothing


## Is the OPT state displacement or velocity?


In [ ]:
# This diagnostic is pure post-processing: it never reruns a solver.
# H_A: stored state q is displacement; dq/dt is velocity.
# H_B: stored state q is velocity; integral(q) is displacement.
function trace_at_station(table, station)
    (time=Float64.(table.time), values=Float64.(table.values[:, station]))
end

function crop_semantics_trace(trace, window)
    selected = findall((trace.time .>= window.start) .&
        (trace.time .<= window.stop))
    length(selected) >= 3 || error("Too few samples for semantics test")
    (time=trace.time[selected], values=trace.values[selected])
end

function integrate_sampled_trace(trace)
    values = zeros(Float64, length(trace.values))
    values[2:end] .= cumsum(
        ((trace.values[1:end-1] .+ trace.values[2:end]) ./ 2) .*
        diff(trace.time))
    (time=Float64.(trace.time), values)
end

compact_shape_metrics(reference, candidate) = begin
    result = waveform_metrics(reference, candidate; samples=501)
    (correlation=result.correlation,
     amplitude_adjusted_error=result.relative_error,
     raw_relative_rmse=result.relative_rmse,
     optimal_amplitude=result.optimal_amplitude)
end

function field_semantics_metrics(raw_table, derivative_table, station, window)
    raw = crop_semantics_trace(trace_at_station(raw_table, station), window)
    derivative = crop_semantics_trace(
        trace_at_station(derivative_table, station), window)
    integrated = integrate_sampled_trace(raw)
    green_u = crop_semantics_trace(
        trace_at_station(greenDisplacementZ, station), window)
    green_v = crop_semantics_trace(
        trace_at_station(greenVelocityZ, station), window)
    hypothesis_displacement = (
        raw_vs_green_displacement=compact_shape_metrics(green_u, raw),
        derivative_vs_green_velocity=compact_shape_metrics(green_v, derivative),
    )
    hypothesis_velocity = (
        raw_vs_green_velocity=compact_shape_metrics(green_v, raw),
        integral_vs_green_displacement=compact_shape_metrics(green_u, integrated),
    )
    score(metrics) = mean(item.amplitude_adjusted_error for item in values(metrics))
    (
        stored_as_displacement=hypothesis_displacement,
        stored_as_velocity=hypothesis_velocity,
        displacement_score=score(hypothesis_displacement),
        velocity_score=score(hypothesis_velocity),
        preferred=score(hypothesis_displacement) <= score(hypothesis_velocity) ?
            :stored_state_is_displacement : :stored_state_is_velocity,
    )
end

fieldSemantics = map(eachindex(receiverX)) do station
    window = safeWindows[station]
    (
        x_km=receiverX[station] / 1e3,
        OPT3=field_semantics_metrics(
            optDisplacementZ, optVelocityZ, station, window),
        convFD3=field_semantics_metrics(
            convFDDisplacementZ, convFDVelocityZ, station, window),
    )
end
foreach(display, fieldSemantics)

fieldSemanticsSummary = map((:OPT3, :convFD3)) do solver
    displacement_score = mean(
        getproperty(item, solver).displacement_score for item in fieldSemantics)
    velocity_score = mean(
        getproperty(item, solver).velocity_score for item in fieldSemantics)
    (
        solver=solver,
        stored_as_displacement_score=displacement_score,
        stored_as_velocity_score=velocity_score,
        preferred=displacement_score <= velocity_score ?
            :stored_state_is_displacement : :stored_state_is_velocity,
    )
end
foreach(display, fieldSemanticsSummary)

# Visual check at the central receiver. Every panel is normalized separately:
# this tests waveform semantics/phase, while optimal_amplitude above tests scale.
station = cld(length(receiverX), 2)
window = safeWindows[station]
rawOPT = crop_semantics_trace(trace_at_station(optDisplacementZ, station), window)
derivativeOPT = crop_semantics_trace(trace_at_station(optVelocityZ, station), window)
integratedOPT = integrate_sampled_trace(rawOPT)
rawConv = crop_semantics_trace(trace_at_station(convFDDisplacementZ, station), window)
derivativeConv = crop_semantics_trace(trace_at_station(convFDVelocityZ, station), window)
integratedConv = integrate_sampled_trace(rawConv)
greenU = crop_semantics_trace(trace_at_station(greenDisplacementZ, station), window)
greenV = crop_semantics_trace(trace_at_station(greenVelocityZ, station), window)

semanticsFigure = Figure(size=(1250, 760))
function normalized_semantics_panel(position, title, reference, opt, conv; ylabel)
    axis = Axis(position; title, xlabel="time (s)", ylabel)
    for (label, trace, color) in (
        ("Green", reference, :black),
        ("OPT3", opt, :dodgerblue),
        ("convFD3", conv, :darkorange),
    )
        scale = max(maximum(abs, trace.values), eps(Float64))
        lines!(axis, trace.time, trace.values ./ scale; label, color)
    end
    axislegend(axis; position=:rb)
    axis
end
normalized_semantics_panel(semanticsFigure[1, 1],
    "H_A: raw state is displacement", greenU, rawOPT, rawConv;
    ylabel="normalized u_z")
normalized_semantics_panel(semanticsFigure[1, 2],
    "H_A: derivative is velocity", greenV, derivativeOPT, derivativeConv;
    ylabel="normalized v_z")
normalized_semantics_panel(semanticsFigure[2, 1],
    "H_B: raw state is velocity", greenV, rawOPT, rawConv;
    ylabel="normalized v_z")
normalized_semantics_panel(semanticsFigure[2, 2],
    "H_B: integral is displacement", greenU, integratedOPT, integratedConv;
    ylabel="normalized u_z")
display(semanticsFigure)
nothing


In [ ]:
function crop_trace(trace, window)
    selected = findall((trace.time .>= window.start) .&
                       (trace.time .<= window.stop))
    length(selected) >= 3 || error("Too few samples in uncontaminated window")
    (time=Float64.(trace.time[selected]),
     values=Float64.(trace.values[selected]))
end

uncontaminatedMetrics = map(eachindex(receiverX)) do station
    window = safeWindows[station]
    greenStation = crop_trace(
        (time=greenWaveformZ.time, values=greenWaveformZ.values[:, station]),
        window)
    fdStation = crop_trace(
        (time=fdWaveformZ.time, values=fdWaveformZ.values[:, station]), window)
    convFDStation = crop_trace(
        (time=convFDWaveformZ.time, values=convFDWaveformZ.values[:, station]),
        window)
    optStation = crop_trace(
        (time=optWaveformZ.time, values=optWaveformZ.values[:, station]), window)
    specfemStation = crop_trace(specfemTraces[station], window)
    greenExplicit = waveform_metrics(greenStation, fdStation)
    greenConv = waveform_metrics(greenStation, convFDStation)
    greenOPT = waveform_metrics(greenStation, optStation)
    greenSPECFEM = waveform_metrics(greenStation, specfemStation)
    (
        x_km=receiverX[station] / 1e3,
        window=(window.start, window.stop),
        convFD3=(correlation=greenConv.correlation,
            relative_error=greenConv.relative_error,
            error_variance=greenConv.error_variance,
            relative_rmse=greenConv.relative_rmse),
        OPT3=(correlation=greenOPT.correlation,
            relative_error=greenOPT.relative_error,
            error_variance=greenOPT.error_variance,
            relative_rmse=greenOPT.relative_rmse),
        SPECFEM2D=(correlation=greenSPECFEM.correlation,
            relative_error=greenSPECFEM.relative_error,
            error_variance=greenSPECFEM.error_variance,
            relative_rmse=greenSPECFEM.relative_rmse),
    )
end
foreach(display, uncontaminatedMetrics)


## Synchronized FD3 / OPT3 propagation video

In [ ]:
nearest_frame(times, time) = argmin(abs.(times .- time))

function record_wavefield_comparison(
    filepath;
    amplitude_mode=:normalized,
    video_dt=0.05,
    framerate=20,
    view_limits=(-35e3, 35e3, -35e3, 20e3),
)
    amplitude_mode in (:normalized, :absolute) ||
        error("amplitude_mode must be :normalized or :absolute")
    frameTimes = collect(0.0:video_dt:min(last(fdTimes),
        last(convFDTimes), last(optTimes)))
    fdPeak = maximum(abs, uzFD)
    convFDPeak = maximum(abs, uzConvFD)
    optPeak = maximum(abs, uzOPT)
    commonPeak = max(fdPeak, convFDPeak, optPeak, eps(Float64))
    fdScale = amplitude_mode === :normalized ? max(fdPeak, eps(Float64)) : 1.0
    convFDScale = amplitude_mode === :normalized ?
        max(convFDPeak, eps(Float64)) : 1.0
    optScale = amplitude_mode === :normalized ? max(optPeak, eps(Float64)) : 1.0
    videoColorRange = amplitude_mode === :normalized ? (-1.0, 1.0) :
        (-commonPeak, commonPeak)
    fdFrame = Observable(Float32.(uzFD[:, :, 1] ./ fdScale))
    convFDFrame = Observable(Float32.(uzConvFD[:, :, 1] ./ convFDScale))
    optFrame = Observable(Float32.(uzOPT[:, :, 1] ./ optScale))
    currentTime = Observable(0.0)
    figure = Figure(size=(1800, 620))
    titleText = amplitude_mode === :normalized ?
        "independent solver normalization" : "common absolute displacement scale"
    fdAxis = Axis(figure[1, 1]; xlabel="x (km)", ylabel="z (km)",
        title=@lift("FD3 — t = $(round($currentTime; digits=2)) s"),
        aspect=DataAspect())
    convFDAxis = Axis(figure[1, 2]; xlabel="x (km)", ylabel="z (km)",
        title=@lift("convFD3 orderB=-1 — t = $(round($currentTime; digits=2)) s"),
        aspect=DataAspect())
    optAxis = Axis(figure[1, 3]; xlabel="x (km)", ylabel="z (km)",
        title=@lift("OPT3 — t = $(round($currentTime; digits=2)) s"),
        aspect=DataAspect())
    Label(figure[0, 1:3], "Vertical displacement: $titleText"; fontsize=22)
    fdPlot = heatmap!(fdAxis, fdCoordinates.x ./ 1e3, fdCoordinates.z ./ 1e3,
        fdFrame; colormap=:balance, colorrange=videoColorRange)
    heatmap!(convFDAxis, collect(xOPT) ./ 1e3, collect(zOPT) ./ 1e3,
        convFDFrame; colormap=:balance, colorrange=videoColorRange)
    heatmap!(optAxis, collect(xOPT) ./ 1e3, collect(zOPT) ./ 1e3,
        optFrame; colormap=:balance, colorrange=videoColorRange)
    if applyFreeSurface
        hlines!(fdAxis, [0.0]; color=:black, linewidth=1.5)
        hlines!(convFDAxis, [0.0]; color=:black, linewidth=1.5)
        hlines!(optAxis, [0.0]; color=:black, linewidth=1.5)
    end
    scatter!(fdAxis, [sourcePosition.x / 1e3], [sourcePosition.z / 1e3];
        marker=:star5, color=:gold, strokecolor=:black, markersize=18)
    scatter!(convFDAxis, [sourcePosition.x / 1e3], [sourcePosition.z / 1e3];
        marker=:star5, color=:gold, strokecolor=:black, markersize=18)
    scatter!(optAxis, [sourcePosition.x / 1e3], [sourcePosition.z / 1e3];
        marker=:star5, color=:gold, strokecolor=:black, markersize=18)
    xminView, xmaxView, zminView, zmaxView = view_limits ./ 1e3
    xlims!(fdAxis, xminView, xmaxView); ylims!(fdAxis, zminView, zmaxView)
    xlims!(convFDAxis, xminView, xmaxView);
    ylims!(convFDAxis, zminView, zmaxView)
    xlims!(optAxis, xminView, xmaxView); ylims!(optAxis, zminView, zmaxView)
    Colorbar(figure[1, 4], fdPlot;
        label=amplitude_mode === :normalized ? "u_z / max|u_z|" :
            "vertical displacement u_z (m)")
    record(figure, filepath, frameTimes; framerate=framerate) do time
        currentTime[] = time
        fdFrame[] = Float32.(uzFD[:, :, nearest_frame(fdTimes, time)] ./ fdScale)
        convFDFrame[] = Float32.(uzConvFD[:, :,
            nearest_frame(convFDTimes, time)] ./ convFDScale)
        optFrame[] = Float32.(uzOPT[:, :, nearest_frame(optTimes, time)] ./ optScale)
    end
    return filepath
end

videoDirectory = joinpath(flexopt_root, "data", "homogeneousElastic2D")
mkpath(videoDirectory)
normalizedVideo = record_wavefield_comparison(
    joinpath(videoDirectory, "explicitFD3_convFD3_OPT3_normalized.mp4");
    amplitude_mode=:normalized,
)
# Set to true when the independently normalized movie looks healthy.
makeAbsoluteVideo = true
absoluteVideo = makeAbsoluteVideo ? record_wavefield_comparison(
    joinpath(videoDirectory, "explicitFD3_convFD3_OPT3_absolute.mp4");
    amplitude_mode=:absolute,
) : nothing
@show normalizedVideo absoluteVideo

# P/S separation from the displacement vector. P is dilatation and S is
# the out-of-plane curl; both are dimensionless strain-like quantities.
function elastic_ps_fields(ux, uz, dx, dz)
    size(ux) == size(uz) || error("uₓ and u_z histories must match")
    p = zeros(Float32, size(ux)); s = similar(p)
    p[2:end-1, 2:end-1, :] .=
        (ux[3:end, 2:end-1, :] .- ux[1:end-2, 2:end-1, :]) ./ (2dx) .+
        (uz[2:end-1, 3:end, :] .- uz[2:end-1, 1:end-2, :]) ./ (2dz)
    s[2:end-1, 2:end-1, :] .=
        (ux[2:end-1, 3:end, :] .- ux[2:end-1, 1:end-2, :]) ./ (2dz) .-
        (uz[3:end, 2:end-1, :] .- uz[1:end-2, 2:end-1, :]) ./ (2dx)
    (; P=p, S=s)
end
psFD = elastic_ps_fields(uxFD, uzFD, step(fdCoordinates.x), step(fdCoordinates.z))
psConvFD = elastic_ps_fields(uxConvFD, uzConvFD, step(xOPT), step(zOPT))
psOPT = elastic_ps_fields(uxOPT, uzOPT, step(xOPT), step(zOPT))
psSPECFEM = elastic_ps_fields(specfemWavefield.ux, specfemWavefield.uz,
    specfemWavefield.x[2]-specfemWavefield.x[1],
    specfemWavefield.z[2]-specfemWavefield.z[1])

function record_ps_comparison(filepath; video_dt=0.05, framerate=20)
    methods = (
        FD3=(x=fdCoordinates.x, z=fdCoordinates.z, time=fdTimes, fields=psFD),
        convFD3=(x=collect(xOPT), z=collect(zOPT), time=convFDTimes, fields=psConvFD),
        OPT3=(x=collect(xOPT), z=collect(zOPT), time=optTimes, fields=psOPT),
        SPECFEM2D=(x=specfemWavefield.x, z=specfemWavefield.z,
            time=specfemWavefield.time, fields=psSPECFEM),
    )
    startTime = maximum(first(method.time) for method in values(methods))
    stopTime = minimum(last(method.time) for method in values(methods))
    frameTimes = collect(startTime:video_dt:stopTime)
    pPeak = maximum(maximum(abs, method.fields.P) for method in values(methods))
    sPeak = maximum(maximum(abs, method.fields.S) for method in values(methods))
    rowPeaks = (P=max(pPeak, eps(Float32)), S=max(sPeak, eps(Float32)))
    currentTime = Observable(startTime)
    figure = Figure(size=(1900, 940))
    observables = Dict{Tuple{Symbol,Symbol},Observable}()
    for (column, (name, method)) in enumerate(pairs(methods))
        for (row, mode) in enumerate((:P, :S))
            frame = Observable(Float32.(getproperty(method.fields, mode)[:, :, 1] ./
                getproperty(rowPeaks, mode)))
            observables[(name, mode)] = frame
            axis = Axis(figure[row, column]; xlabel="x (km)", ylabel="z (km)",
                title=@lift("$(name) $(mode) — t=$(round($currentTime; digits=2)) s"),
                aspect=DataAspect())
            heatmap!(axis, method.x ./ 1e3, method.z ./ 1e3, frame;
                colormap=:balance, colorrange=(-1, 1))
            scatter!(axis, [sourcePosition.x/1e3], [sourcePosition.z/1e3];
                marker=:star5, color=:gold, strokecolor=:black, markersize=14)
        end
    end
    Label(figure[0, 1:4],
        "P = ∂ₓuₓ + ∂zu_z; S = ∂zuₓ - ∂ₓu_z — common scale within each row";
        fontsize=21)
    record(figure, filepath, frameTimes; framerate=framerate) do time
        currentTime[] = time
        for (name, method) in pairs(methods), mode in (:P, :S)
            frame = nearest_frame(method.time, time)
            observables[(name, mode)][] = Float32.(
                getproperty(method.fields, mode)[:, :, frame] ./
                getproperty(rowPeaks, mode))
        end
    end
    filepath
end
psVideo = record_ps_comparison(joinpath(videoDirectory,
    "FD3_convFD3_OPT3_SPECFEM2D_P_S.mp4"))
@show psVideo
psVideo


## Interpretation

Arrival times and waveform correlation should be checked before raw amplitude. All three runs use the same nominal 2-D vertical line force, but FD mass lumping, OPT weak/source quadrature and SPECFEM GLL source projection are distinct discretizations. A stable peak ratio under grid refinement is therefore the useful absolute-amplitude test.